# Experiment 5 — Staged unfreeze: 4 epochs frozen + 8 epochs unfrozen on `dataset_matan`

Warm-starts from **`cyttic/trocr-hebrew-untrained`** and finetunes on
**`cyttic/trocr-hebrew-matan`** in **two phases, 12 epochs total**:

| Phase | Epochs | Encoder | Why |
|---|---|---|---|
| 1 | **1-4** | **frozen** | let the random cross-attention + decoder learn to read against *stable*, known-good ViT features first |
| 2 | **5-12** | **unfrozen** | once cross-attention is no longer noise, let the encoder adapt to real handwritten ink/strokes |

**Why this design:** Experiment 4 (unfrozen from epoch 1, random cross-attention)
showed CER bouncing around ~0.8-1.0 in the early epochs — plausibly because
gradients from a still-random cross-attention were pushing the pretrained
encoder's good visual features in random directions before cross-attention had
learned anything useful. This staged schedule mirrors the project's own
validated recipe (exp2: frozen synthetic pretrain -> exp3: unfrozen continuation),
just compressed into a single run with two back-to-back `Trainer` phases.

Each phase is its **own** `Seq2SeqTrainer` with its own optimizer (changing
`requires_grad` mid-run isn't something the HF `Trainer` supports cleanly — a
fresh optimizer matched to the new trainable-parameter set is the standard,
robust way to do staged unfreezing). Phase 2 warm-starts from Phase 1's final
weights.

**No pre-train baseline this time** (per your call — only metrics *during*
training matter here): in-training CER/WER come from `compute_metrics` each
epoch, and a single full beam-search CER/WER/BLEU pass runs at the very end.

**Target hardware:** L4 (24 GB, bf16). Checkpoints push to the Hub **once per
epoch** in both phases (`save_strategy="epoch"`), so a lost Colab session can
resume — re-run top to bottom and each phase picks up where it left off.

## 1. Setup

Self-contained: downloads the model + dataset from HuggingFace. Run top to bottom.

In [ ]:
# Run once on a fresh VM/Colab runtime. Comment out if deps are already installed.
!pip install -q torch transformers datasets accelerate jiwer sacrebleu pillow matplotlib

In [ ]:
import os
import torch
import jiwer
import sacrebleu
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    VisionEncoderDecoderModel,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint
from huggingface_hub import snapshot_download
from huggingface_hub.utils import RepositoryNotFoundError

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("vram  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    print("bf16  :", torch.cuda.is_bf16_supported())

In [ ]:
# --- HuggingFace auth (Colab has no persistent box auth -> log in every session) ---
# Needs a WRITE token. Create one at https://huggingface.co/settings/tokens (token type: Write)
from huggingface_hub import notebook_login, whoami
notebook_login()
print("logged in as:", whoami()["name"])

In [ ]:
# --- HebrewBlockProcessor (inlined so this notebook is self-contained) ---
from PIL import Image, ImageOps

class HebrewBlockProcessor:
    """Mirror (RTL->LTR) -> resize to 64px height -> tile into a 384x384 ViT container."""
    TARGET_HEIGHT = 64
    CONTAINER_SIZE = 384
    IMAGE_MEAN = [0.5, 0.5, 0.5]
    IMAGE_STD = [0.5, 0.5, 0.5]

    def __call__(self, images, return_tensors="pt"):
        if not isinstance(images, list):
            images = [images]
        pixel_values = torch.stack([self._process(img) for img in images])
        return {"pixel_values": pixel_values}

    def _process(self, image):
        image = image.convert("RGB")
        image = ImageOps.mirror(image)
        w, h = image.size
        new_w = max(1, round(w * self.TARGET_HEIGHT / h))
        image = image.resize((new_w, self.TARGET_HEIGHT), Image.LANCZOS)
        container = Image.new("RGB", (self.CONTAINER_SIZE, self.CONTAINER_SIZE), (255, 255, 255))
        img_arr = np.array(image)
        src_x, dest_x, dest_y = 0, 0, 0
        while src_x < new_w and dest_y < self.CONTAINER_SIZE:
            chunk_w = min(new_w - src_x, self.CONTAINER_SIZE - dest_x)
            chunk = Image.fromarray(img_arr[:, src_x:src_x + chunk_w])
            container.paste(chunk, (dest_x, dest_y))
            src_x += chunk_w
            dest_x += chunk_w
            if dest_x >= self.CONTAINER_SIZE:
                dest_x = 0
                dest_y += self.TARGET_HEIGHT
        t = torch.tensor(np.array(container), dtype=torch.float32).permute(2, 0, 1) / 255.0
        mean = torch.tensor(self.IMAGE_MEAN).view(3, 1, 1)
        std = torch.tensor(self.IMAGE_STD).view(3, 1, 1)
        return (t - mean) / std

## 2. Config

Two LRs on purpose: `FROZEN_LR` matches `train.py`'s default for finetuning the
untrained model with fixed visual features; `UNFROZEN_LR` is lower, matching
exp3's reasoning that an unfrozen encoder is more sensitive to the learning rate.

In [ ]:
MODEL_ID   = "cyttic/trocr-hebrew-untrained"   # ViT-handwritten encoder + DictaBERT decoder, random cross-attn
DATASET_ID = "cyttic/trocr-hebrew-matan"        # writer-level train/test split (see to_parquet_pairs.py)

FROZEN_EPOCHS   = 4    # Phase 1: encoder frozen   -- epochs 1-4
UNFROZEN_EPOCHS = 8    # Phase 2: encoder unfrozen -- epochs 5-12  (12 total)

# Phase 1 (frozen encoder): gradients/optimizer states only cover ~213M of the
# 299.5M params, so batch 16 fits comfortably (matches the project's frozen runs).
BATCH_SIZE        = 16
GRAD_ACCUM        = 1

# Phase 2 (unfrozen encoder): now the FULL 299.5M params need gradients + AdamW
# m/v states + backprop activations -- batch 16 OOMs immediately on a 24GB L4.
# exp3's README hit the exact same wall and dropped to batch 8 for its unfrozen
# run; we do the same here, and use grad-accum 2 to keep the *effective* batch
# size at 16 so the LR schedule behaves the same as if nothing changed.
UNFROZEN_BATCH_SIZE = 8
UNFROZEN_GRAD_ACCUM = 2

FROZEN_LR         = 5e-5   # phase 1 -- matches train.py's default for the frozen-encoder finetune
UNFROZEN_LR       = 3e-5   # phase 2 -- decoder + cross-attention LR (these still need to move a lot)

# Discriminative LR for phase 2: the encoder is *pretrained* (good visual
# features) and was only just unfrozen, while cross-attention has had just 4
# epochs to get out of "pure noise" territory. A single shared LR lets the
# still-noisy cross-attention gradients yank the encoder's good features around
# -- exactly the val-loss-creeps-up-while-train-loss-falls symptom seen at the
# epoch 5-6 transition. Giving the encoder ~10x less LR lets it adapt gently
# while cross-attn/decoder keep moving at full speed (ULMFiT-style gradual
# unfreezing -- standard fix for "pretrained backbone + freshly-unfrozen / still
# -randomish heads sharing one optimizer").
ENCODER_LR        = UNFROZEN_LR / 10   # 3e-6

MAX_TARGET_LENGTH = 128
NUM_WORKERS       = 4
PRECISION         = "bf16" # target hardware: L4 (24GB, bf16). Use "fp16" on a Turing card (RTX 2080).
MAX_STEPS         = -1     # set e.g. 50 for a quick smoke test of EITHER phase

RUN_NAME   = "trocr-hebrew-matan-staged"
OUTPUT_DIR = f"output/{RUN_NAME}"

PHASE1_DIR        = f"{OUTPUT_DIR}/phase1_frozen"
HUB_PHASE1_CKPTS  = f"cyttic/{RUN_NAME}-phase1-ckpts"   # phase-1 mid-run checkpoints (resume)
HUB_PHASE1_REPO   = f"cyttic/{RUN_NAME}-phase1"         # phase-1 FINAL weights (durable handoff to phase 2)
PHASE1_FINAL_DIR  = f"{PHASE1_DIR}/final"

PHASE2_DIR    = f"{OUTPUT_DIR}/phase2_unfrozen"
HUB_CKPTS_REPO = f"cyttic/{RUN_NAME}-ckpts"             # phase-2 mid-run checkpoints (resume)
HUB_REPO       = f"cyttic/{RUN_NAME}"                   # FINAL model of the whole experiment

SAVE_LIMIT = 3   # keep only the last N local checkpoints per phase

print(f"phase 1: {FROZEN_EPOCHS} epochs, encoder FROZEN,   lr={FROZEN_LR}, batch={BATCH_SIZE}x{GRAD_ACCUM}")
print(f"phase 2: {UNFROZEN_EPOCHS} epochs, encoder UNFROZEN, decoder/cross-attn lr={UNFROZEN_LR}, "
      f"encoder lr={ENCODER_LR:g}, batch={UNFROZEN_BATCH_SIZE}x{UNFROZEN_GRAD_ACCUM}")
print("phase-1 dir  :", PHASE1_DIR,  "| ckpts:", HUB_PHASE1_CKPTS, "| final repo:", HUB_PHASE1_REPO)
print("phase-2 dir  :", PHASE2_DIR,  "| ckpts:", HUB_CKPTS_REPO)
print("final repo   :", HUB_REPO)

## 3. Model, tokenizer, processor

Loaded fresh from the **untrained** base. Freezing/unfreezing happens explicitly at the start of each phase below.

In [ ]:
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
processor = HebrewBlockProcessor()

# generation_config takes priority over model.config in recent transformers,
# so set the special tokens on it explicitly or generate() fails during eval.
def reset_generation_config(m):
    m.generation_config.decoder_start_token_id = tokenizer.cls_token_id
    m.generation_config.pad_token_id = tokenizer.pad_token_id
    m.generation_config.eos_token_id = tokenizer.sep_token_id
    m.generation_config.max_new_tokens = None

reset_generation_config(model)

def report_trainable(m, tag=""):
    n_total = sum(p.numel() for p in m.parameters())
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{tag} params total: {n_total/1e6:.1f}M | trainable: {n_train/1e6:.1f}M "
          f"({100*n_train/n_total:.1f}%)")

report_trainable(model, "[loaded]")

## 4. Dataset

In [ ]:
ds = load_dataset(DATASET_ID)
print(ds)

eval_ds = ds["test"]
print("eval (test) samples:", len(eval_ds))

## 5. Sanity checks

Confirm the `train`/`test` split is **writer-level / leak-free** (no writer's
lines appear in both, the same principle `CLAUDE.md` requires for the human
set), then *see* what `HebrewBlockProcessor` does to a real line.

In [ ]:
tr_writers = set(ds["train"]["writer"])
te_writers = set(ds["test"]["writer"])
print(f"train writers: {len(tr_writers)} | test writers: {len(te_writers)} | "
      f"OVERLAP: {len(tr_writers & te_writers)} (should be 0)")

In [ ]:
sample = ds["train"][0]
img = sample["image"].convert("RGB")
print("writer:", sample["writer"], "| text:", sample["text"])

pv = processor([img])["pixel_values"][0]
shown = (pv * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].imshow(img);   ax[0].set_title("raw matan line"); ax[0].axis("off")
ax[1].imshow(shown); ax[1].set_title("after HebrewBlockProcessor (mirrored + tiled 384x384)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Collator + metrics (CER / WER, used by both phases' Trainer)

In [ ]:
def collate(batch):
    images = [ex["image"].convert("RGB") for ex in batch]
    texts  = [ex["text"] for ex in batch]
    pixel_values = processor(images)["pixel_values"]
    labels = tokenizer(
        texts, padding="longest", truncation=True,
        max_length=MAX_TARGET_LENGTH, return_tensors="pt",
    ).input_ids
    labels[labels == tokenizer.pad_token_id] = -100
    return {"pixel_values": pixel_values, "labels": labels}


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    pred_ids  = np.where(pred_ids  < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"cer": jiwer.cer(label_str, pred_str),
            "wer": jiwer.wer(label_str, pred_str)}


def make_targs(output_dir, epochs, lr, hub_ckpts_repo, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM):
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        num_train_epochs=epochs,
        max_steps=MAX_STEPS,
        bf16=(PRECISION == "bf16"),
        fp16=(PRECISION == "fp16"),
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        generation_num_beams=1,
        eval_strategy="epoch",
        save_strategy="epoch",         # checkpoint (and push to the Hub) once per epoch
        logging_steps=50,
        save_total_limit=SAVE_LIMIT,
        load_best_model_at_end=False,  # keep the latest epoch's weights, not "best by eval CER"
        dataloader_num_workers=NUM_WORKERS,
        remove_unused_columns=False,   # keep image/text for the custom collator
        report_to="none",
        push_to_hub=True,
        hub_model_id=hub_ckpts_repo,
        hub_strategy="checkpoint",     # only the latest checkpoint kept on the Hub (for resume)
        hub_private_repo=True,
    )


def resolve_resume_checkpoint(local_dir, hub_ckpts_repo):
    """Local checkpoint first; else pull 'last-checkpoint' from the Hub (Colab has no durable disk)."""
    ckpt = get_last_checkpoint(local_dir) if os.path.isdir(local_dir) else None
    if ckpt is None:
        try:
            snapshot_download(hub_ckpts_repo, repo_type="model",
                              local_dir=local_dir, allow_patterns="last-checkpoint/*")
            cand = os.path.join(local_dir, "last-checkpoint")
            ckpt = cand if os.path.isdir(cand) else None
        except RepositoryNotFoundError:
            pass  # no checkpoints pushed yet
    return ckpt

## 7. Phase 1 — encoder FROZEN (epochs 1-4)

Freeze the ViT encoder so the random cross-attention + decoder learn to read
against *stable*, known-good visual features first. Checkpoints push to
`HUB_PHASE1_CKPTS` once per epoch; the final phase-1 weights are saved locally
to `PHASE1_FINAL_DIR` **and** pushed to `HUB_PHASE1_REPO` so phase 2 can pick
them up even from a fresh Colab session.

If `PHASE1_FINAL_DIR` already exists (e.g. you already ran phase 1 and are
re-running the notebook), this cell skips straight to loading those weights.

In [ ]:
if os.path.isdir(PHASE1_FINAL_DIR):
    print(f"Phase 1 already complete -- loading final weights from {PHASE1_FINAL_DIR}")
    model = VisionEncoderDecoderModel.from_pretrained(PHASE1_FINAL_DIR)
    reset_generation_config(model)
else:
    for p in model.encoder.parameters():
        p.requires_grad = False
    report_trainable(model, "[phase 1 / frozen]")

    targs1 = make_targs(PHASE1_DIR, FROZEN_EPOCHS, FROZEN_LR, HUB_PHASE1_CKPTS)
    trainer1 = Seq2SeqTrainer(
        model=model, args=targs1,
        train_dataset=ds["train"], eval_dataset=eval_ds,
        data_collator=collate, compute_metrics=compute_metrics,
    )

    last_ckpt = resolve_resume_checkpoint(PHASE1_DIR, HUB_PHASE1_CKPTS)
    print("Resuming phase 1 from", last_ckpt) if last_ckpt else print("Starting phase 1 from the untrained warm-start weights")
    trainer1.train(resume_from_checkpoint=last_ckpt)

    trainer1.save_model(PHASE1_FINAL_DIR)
    tokenizer.save_pretrained(PHASE1_FINAL_DIR)
    print("phase 1 saved ->", PHASE1_FINAL_DIR)

    model.push_to_hub(HUB_PHASE1_REPO)
    tokenizer.push_to_hub(HUB_PHASE1_REPO)
    print("phase 1 pushed ->", HUB_PHASE1_REPO)

## 8. Phase 2 — encoder UNFROZEN (epochs 5-12)

Warm-starts from the phase-1 weights (loaded above, or pulled fresh from
`HUB_PHASE1_REPO` if this is a new session and the local copy is gone), then
**unfreezes the encoder** and trains for `UNFROZEN_EPOCHS` more epochs with a
**new** `Trainer`/optimizer (HF `Trainer` doesn't support changing
`requires_grad` mid-run cleanly — a fresh optimizer matched to the new
trainable-parameter set is the standard way to do staged unfreezing).
Checkpoints push to `HUB_CKPTS_REPO` once per epoch.

In [ ]:
import gc

# Drop everything from the aborted single-LR phase-2 attempt (trainer, optimizer,
# the drifted in-memory model) -- its CUDA tensors would otherwise crowd out the
# fresh run, and its weights already show the bad val-loss/CER trend we're fixing.
for name in ("trainer1", "trainer2", "optimizer", "targs2", "last_ckpt"):
    if name in dir():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# Reload phase-1's weights FRESH -- the in-memory `model` already drifted through
# ~2 epochs of the single-LR attempt (the one whose val loss/CER ticked the wrong
# way right at the unfreeze point). PHASE1_FINAL_DIR / HUB_PHASE1_REPO hold the
# untouched phase-1 hand-off, so this restart costs nothing real.
print("Reloading phase-1 weights fresh (discarding the single-LR phase-2 attempt)")
model = VisionEncoderDecoderModel.from_pretrained(
    PHASE1_FINAL_DIR if os.path.isdir(PHASE1_FINAL_DIR) else HUB_PHASE1_REPO
)
reset_generation_config(model)

for p in model.encoder.parameters():
    p.requires_grad = True
report_trainable(model, "[phase 2 / unfrozen, discriminative LR]")

# Discriminative LR -- two param groups so the (pretrained, just-unfrozen)
# encoder moves ~10x slower than cross-attention/decoder. This directly targets
# the epoch 5-6 symptom: noisy gradients from a still-settling cross-attention
# yanking the encoder's good visual features around (val loss creeping up while
# train loss fell, CER/WER ticking the wrong way). A shared LR can't fix that;
# gentler encoder updates can -- standard "gradual unfreezing" practice.
encoder_params = [p for n, p in model.named_parameters() if n.startswith("encoder.") and p.requires_grad]
other_params   = [p for n, p in model.named_parameters() if not n.startswith("encoder.") and p.requires_grad]
print(f"optimizer groups -- encoder: {sum(p.numel() for p in encoder_params)/1e6:.1f}M @ lr={ENCODER_LR:g} | "
      f"decoder+cross-attn: {sum(p.numel() for p in other_params)/1e6:.1f}M @ lr={UNFROZEN_LR:g}")

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": ENCODER_LR},
    {"params": other_params,   "lr": UNFROZEN_LR},
], weight_decay=0.01)

targs2 = make_targs(PHASE2_DIR, UNFROZEN_EPOCHS, UNFROZEN_LR, HUB_CKPTS_REPO,
                    batch_size=UNFROZEN_BATCH_SIZE, grad_accum=UNFROZEN_GRAD_ACCUM)
trainer2 = Seq2SeqTrainer(
    model=model, args=targs2,
    train_dataset=ds["train"], eval_dataset=eval_ds,
    data_collator=collate, compute_metrics=compute_metrics,
    optimizers=(optimizer, None),   # our 2-group AdamW; Trainer still builds the default LR scheduler around it
)

# Starting fresh on purpose, NOT resuming: the old phase-2 checkpoint's optimizer
# state was built for a single shared-LR AdamW and is structurally incompatible
# with the 2-group optimizer above (loading it would error or silently misapply
# LRs). It'll be naturally replaced once this run pushes its first new checkpoint
# (hub_strategy="checkpoint" keeps only the latest). If THIS run gets interrupted
# later, re-running this cell will reload from PHASE1_FINAL_DIR again -- to resume
# mid-run instead, swap in resolve_resume_checkpoint(PHASE2_DIR, HUB_CKPTS_REPO)
# once at least one epoch has checkpointed under the new optimizer.
print("Starting phase 2 fresh from the phase-1 weights with discriminative LRs (encoder now unfrozen)")
trainer2.train()

## 9. Final CER / WER / BLEU (full beam-search pass)

`trainer.evaluate()` only reports CER/WER (greedy by default), so BLEU needs its
own beam-search generation pass over the full matan test split — this is the
single metrics readout that matters for judging the finished model (per your
call, no pre-train baseline this time).

In [ ]:
@torch.no_grad()
def eval_cer_wer_bleu(model, dataset, beams=4, batch_size=BATCH_SIZE, max_length=MAX_TARGET_LENGTH, tag=""):
    model.eval()
    refs, hyps = [], []
    for start in range(0, len(dataset), batch_size):
        batch = dataset[start:start + batch_size]
        imgs = [im.convert("RGB") for im in batch["image"]]
        pv = processor(imgs)["pixel_values"].to(model.device, dtype=model.dtype)
        ids = model.generate(pv, num_beams=beams, max_length=max_length)
        hyps.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
        refs.extend(batch["text"])
        done = min(start + batch_size, len(dataset))
        if done % (batch_size * 5) == 0 or done == len(dataset):
            print(f"  {tag} {done}/{len(dataset)}", flush=True)

    cer = jiwer.cer(refs, hyps)
    wer = jiwer.wer(refs, hyps)
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    exact = sum(r.strip() == h.strip() for r, h in zip(refs, hyps)) / len(refs)
    return {"cer": cer, "wer": wer, "bleu": bleu, "exact": exact, "refs": refs, "hyps": hyps}


print(f"Running FINAL eval (12-epoch staged model) on {len(eval_ds)} matan test lines, beam=4 ...")
final = eval_cer_wer_bleu(model, eval_ds, beams=4, tag="[final]")
print(f"\nFINAL (4 frozen + 8 unfrozen epochs)  |  N={len(eval_ds)}  |  "
      f"CER {final['cer']*100:.2f}%  WER {final['wer']*100:.2f}%  "
      f"BLEU {final['bleu']:.2f}  |  exact {final['exact']*100:.2f}%")

## 10. Look at predictions

In [ ]:
model.eval()
n = 6
fig, axes = plt.subplots(n, 1, figsize=(10, 2.2 * n))
for ax, ex in zip(axes, ds["test"].select(range(n))):
    img = ex["image"].convert("RGB")
    pv = processor([img])["pixel_values"].to(model.device, dtype=model.dtype)
    with torch.no_grad():
        ids = model.generate(pv, num_beams=4, max_new_tokens=MAX_TARGET_LENGTH)
    pred = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"GT  : {ex['text']}\nPRED: {pred}", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## 11. Save & push the final model

Pushes the finished (4 frozen + 8 unfrozen epoch) model to `cyttic/trocr-hebrew-matan-staged`.

In [ ]:
final_dir = f"{OUTPUT_DIR}/final"
trainer2.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("saved ->", final_dir)

model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print("pushed ->", HUB_REPO)